# Trustworthy Credit Scoring with LightGBM, SHAP, LIME & Bias Audit
### German Credit Dataset (UCI) · XAI Pipeline

**Author:** Sourour Hammoud  
**Notebook structure:**

| Cell | Topic |
|------|-------|
| 0 | Library installation |
| 1 | Imports & setup |
| 2 | Step 1 – Load German Credit data |
| 3 | Step 2 – Pre-processing |
| 4 | Step 3 – Class imbalance (SMOTE + scale_pos_weight) |
| 5 | Step 4 – Train LightGBM |
| 6 | Step 5 – Model evaluation |
| 7 | Step 6 – XAI: SHAP (global + local) |
| 8 | Step 7 – XAI: LIME (local surrogate) |
| 9 | Step 8 – Bias audit via SHAP |
| 10 | Step 9 – Project summary figure |


## Cell 0 · Library Installation
Install all third-party packages that are not included in a standard Colab environment.  
`imbalanced-learn` provides SMOTE; `lightgbm` is the gradient-boosted tree framework;
`shap` and `lime` are the two XAI libraries used later.


In [ ]:
# Run once per Colab session
!pip install imbalanced-learn lightgbm shap lime --quiet


## Cell 1 · Imports & Setup
All imports are collected in one place for readability.
- **numpy / pandas** – numerical and tabular data handling  
- **matplotlib / seaborn** – visualisation  
- **sklearn** – train/test split, label encoding, metrics  
- **imblearn** – SMOTE over-sampler  
- **lightgbm** – gradient-boosted tree classifier  
- **shap** – Shapley-value-based global/local explanations  
- **lime** – local surrogate model explanations  


In [ ]:
import warnings, os
warnings.filterwarnings("ignore")

import numpy  as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection   import train_test_split
from sklearn.preprocessing     import LabelEncoder
from sklearn.metrics           import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score, f1_score
)

from imblearn.over_sampling    import SMOTE
import lightgbm as lgb
import shap

try:
    import lime
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "lime", "-q"])
    import lime
import lime.lime_tabular

# Output directory
OUT = "/mnt/user-data/outputs"
os.makedirs(OUT, exist_ok=True)
print("All libraries loaded successfully.")


## Cell 2 · Step 1 — Load German Credit Dataset (UCI)

The **German Credit** dataset contains 1 000 loan applicants described by 20 attributes
(checking account status, loan duration, credit history, purpose, savings, employment, etc.).
The target label is binary:

- **0 = Good** credit (the bank would approve)  
- **1 = Default / Bad** (the bank should reject)

**Class imbalance:** 700 good vs 300 bad → 2.3 : 1 ratio.  
This moderate imbalance must be handled explicitly to avoid the model ignoring the minority class.


In [ ]:
col_names = [
    "checking_status","duration","credit_history","purpose","credit_amount",
    "savings_status","employment","installment_commitment","personal_status",
    "other_parties","residence_since","property_magnitude","age",
    "other_payment_plans","housing","existing_credits","job",
    "num_dependents","own_telephone","foreign_worker","class"
]

url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
       "statlog/german/german.data")

df = pd.read_csv(url, sep=" ", header=None, names=col_names)

# Remap: 1=Good→0, 2=Bad→1
df["target"] = (df["class"] == 2).astype(int)
df.drop(columns=["class"], inplace=True)

print(f"Loaded: {df.shape[0]} rows, {df.shape[1]} cols")
vc = df["target"].value_counts()
for k, v in vc.items():
    label = "Default (BAD)" if k == 1 else "Good"
    print(f"  {k} — {label:<18}  {v:>4}  ({v/len(df)*100:.1f}%)")

imbalance_ratio = vc[0] / vc[1]
print(f"\nImbalance ratio  Good:Bad = {imbalance_ratio:.1f}:1  →  MODERATE IMBALANCE")


## Cell 3 · Step 2 — Pre-processing

**Steps performed:**
1. Separate features `X` from target `y`.
2. Identify categorical vs numeric columns.
3. **Label-encode** all categorical features.  
   LightGBM supports integer-encoded categoricals natively; this avoids the dimensionality explosion of one-hot encoding.
4. **Stratified train/test split** (80 % / 20 %) to preserve the class ratio in both sets.

> No scaling is applied because LightGBM is a tree-based model and is invariant to monotone feature transformations.


In [ ]:
X = df.drop(columns=["target"])
y = df["target"]

cat_cols = X.select_dtypes(include="object").columns.tolist()
num_cols = X.select_dtypes(exclude="object").columns.tolist()
print(f"Categorical features : {len(cat_cols)}")
print(f"Numeric  features    : {len(num_cols)}")

le_map = {}
for c in cat_cols:
    le = LabelEncoder()
    X[c] = le.fit_transform(X[c].astype(str))
    le_map[c] = le

feature_names = X.columns.tolist()
print(f"Total features after encoding : {len(feature_names)}")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)
print(f"\nTrain : {X_train.shape[0]}   Test : {X_test.shape[0]}")
print(f"Train default rate : {y_train.mean()*100:.1f}%   "
      f"Test default rate : {y_test.mean()*100:.1f}%")


## Cell 4 · Step 3 — Handling Class Imbalance

A **two-pronged strategy** is used:

### A) SMOTE — Synthetic Minority Over-sampling Technique
SMOTE generates *synthetic* minority-class samples in feature space by interpolating
between existing minority instances and their k nearest neighbours (`k=5`).

- Applied **only to training data** (the test set is never touched — this prevents data leakage).
- After SMOTE the training set is balanced 1 : 1 (560 good : 560 default).

### B) `scale_pos_weight` in LightGBM
Sets a cost weight = majority_count / minority_count so the model penalises
false-negatives (missed defaults) more heavily than false-positives (unnecessary rejections).
This matches real-world bank cost asymmetry.

> Using both methods together provides complementary coverage: SMOTE works at the
> data level; `scale_pos_weight` works at the loss function level.


In [ ]:
smote = SMOTE(random_state=42, k_neighbors=5)
X_res, y_res = smote.fit_resample(X_train, y_train)

print(f"Before SMOTE → Train positives: {y_train.sum()}  negatives: {(y_train==0).sum()}")
print(f"After  SMOTE → Train positives: {y_res.sum()}  negatives: {(y_res==0).sum()}")

spw = (y_train == 0).sum() / (y_train == 1).sum()
print(f"scale_pos_weight = {spw:.2f}")


## Cell 5 · Step 4 — Training LightGBM Classifier

**Why LightGBM?**
- Gradient-boosted decision trees with **leaf-wise growth** → better accuracy than level-wise (XGBoost default) on tabular data.
- **Histogram-based binning** → 10–20× faster training, lower memory footprint.
- Native categorical feature support and built-in feature importance.
- `TreeExplainer` (SHAP) is O(T·L·D) — extremely fast on LightGBM models.
- `scale_pos_weight` parameter directly addresses class imbalance.

**Key hyperparameters:**

| Parameter | Value | Reason |
|-----------|-------|--------|
| `n_estimators` | 500 | Max rounds; early stopping selects best |
| `learning_rate` | 0.05 | Slow learning → better generalisation |
| `max_depth` | 6 | Prevents very deep trees (overfitting) |
| `num_leaves` | 31 | Controls model complexity |
| `subsample` | 0.8 | Row subsampling → regularisation |
| `colsample_bytree` | 0.8 | Column subsampling → regularisation |
| `reg_alpha / reg_lambda` | 0.1 | L1 / L2 weight regularisation |

Early stopping halts training when the validation metric does not improve for 50 rounds.


In [ ]:
model = lgb.LGBMClassifier(
    n_estimators        = 500,
    learning_rate       = 0.05,
    max_depth           = 6,
    num_leaves          = 31,
    min_child_samples   = 20,
    subsample           = 0.8,
    colsample_bytree    = 0.8,
    reg_alpha           = 0.1,
    reg_lambda          = 0.1,
    scale_pos_weight    = spw,
    random_state        = 42,
    n_jobs              = -1,
    verbose             = -1
)

model.fit(
    X_res, y_res,
    eval_set            = [(X_test, y_test)],
    callbacks           = [lgb.early_stopping(50, verbose=False),
                           lgb.log_evaluation(-1)]
)

best_iter = model.best_iteration_
print(f"Training complete  —  best iteration: {best_iter}")


## Cell 6 · Step 5 — Model Evaluation

Three complementary metrics are reported:

| Metric | What it measures | Why it matters here |
|--------|-----------------|---------------------|
| **ROC-AUC** | Ranking quality across all thresholds | Standard; robust to imbalance |
| **Average Precision** | Area under Precision-Recall curve | Better than AUC when positives are rare |
| **F1 Score** | Harmonic mean of precision & recall | Single-number operational quality |

**Three plots are produced:**
- **ROC curve** – shows the trade-off between true positive rate and false positive rate.
- **Precision-Recall curve** – more informative than ROC when classes are imbalanced.
- **Confusion matrix** – shows exact counts of TP, FP, TN, FN at threshold = 0.50.


In [ ]:
y_pred_proba = model.predict_proba(X_test)[:, 1]
y_pred       = (y_pred_proba >= 0.50).astype(int)

roc_auc = roc_auc_score(y_test, y_pred_proba)
ap      = average_precision_score(y_test, y_pred_proba)
f1      = f1_score(y_test, y_pred)

print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"Avg Prec : {ap:.4f}")
print(f"F1 Score : {f1:.4f}")
print(f"\nClassification Report (threshold=0.50):")
print(classification_report(y_test, y_pred, target_names=["Good (0)","Default (1)"]))

# ── Evaluation dashboard ──────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("LightGBM — Model Evaluation Dashboard", fontsize=14, fontweight="bold")

fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0].plot(fpr, tpr, color="#2563EB", lw=2.5, label=f"LightGBM (AUC = {roc_auc:.3f})")
axes[0].plot([0,1],[0,1],"--", color="grey", lw=1)
axes[0].fill_between(fpr, tpr, alpha=0.08, color="#2563EB")
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate", title="ROC Curve")
axes[0].legend(loc="lower right"); axes[0].grid(alpha=0.3)

prec, rec, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1].plot(rec, prec, color="#DC2626", lw=2.5, label=f"AP = {ap:.3f}")
axes[1].axhline(y_test.mean(), ls="--", color="grey", label=f"Baseline ({y_test.mean():.2f})")
axes[1].fill_between(rec, prec, alpha=0.08, color="#DC2626")
axes[1].set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curve")
axes[1].legend(); axes[1].grid(alpha=0.3)

cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=axes[2],
            xticklabels=["Good","Default"], yticklabels=["Good","Default"],
            linewidths=0.5, cbar=False)
axes[2].set(xlabel="Predicted", ylabel="Actual", title="Confusion Matrix")

plt.tight_layout()
plt.savefig(f"{OUT}/01_evaluation_dashboard.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved evaluation dashboard")


## Cell 7 · Step 6 — XAI: SHAP (SHapley Additive exPlanations)

### Theory
SHAP is grounded in **cooperative game theory** (Shapley, 1953). For each prediction,
it asks: *"How much did each feature contribute to pushing the model output away from the average prediction?"*

**Four axiomatic properties** (guarantees no other method satisfies simultaneously):

| Property | Meaning |
|----------|---------|
| **Consistency** | If a feature's importance increases in a model, its SHAP value never decreases |
| **Local accuracy** | SHAP values sum exactly to f(x) − E[f(x)] |
| **Missingness** | Features that are absent get SHAP = 0 |
| **TreeExplainer** | Exact O(T·L·D) algorithm for tree models — no sampling, no approximation |

### Three visualisations produced
| Plot | Type | What it shows |
|------|------|---------------|
| **(A) Global Bar** | Global | Mean \|SHAP\| per feature — overall importance ranking |
| **(B) Beeswarm** | Global | Full distribution of SHAP values — direction + magnitude per sample |
| **(C) Waterfall** | Local | Single rejected applicant: step-by-step contribution breakdown |


In [ ]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# Binary classification: take class-1 (Default) SHAP values
if isinstance(shap_values, list):
    sv = shap_values[1]
else:
    sv = shap_values

# ── (A) Global SHAP Bar ────────────────────────────────────────────────────
mean_abs_shap = np.abs(sv).mean(axis=0)
sorted_idx    = np.argsort(mean_abs_shap)[::-1]
top_n         = 15

fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#2563EB" if i < 5 else "#93C5FD" for i in range(top_n)]
bars   = ax.barh(range(top_n),
                 mean_abs_shap[sorted_idx[:top_n]][::-1],
                 color=colors[::-1], edgecolor="white", linewidth=0.5)
ax.set_yticks(range(top_n))
ax.set_yticklabels([feature_names[i] for i in sorted_idx[:top_n]][::-1], fontsize=10)
ax.set_xlabel("Mean |SHAP Value| (impact on model output)", fontsize=11)
ax.set_title("Global Feature Importance — SHAP Bar Plot\n"
             "Higher = feature more influential overall", fontsize=12, fontweight="bold")
ax.grid(axis="x", alpha=0.3)
for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.001, bar.get_y() + bar.get_height()/2,
            f"{w:.4f}", va="center", fontsize=8)
plt.tight_layout()
plt.savefig(f"{OUT}/02A_shap_global_bar.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved SHAP global bar")

# ── (B) Beeswarm ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 8))
shap_exp = shap.Explanation(
    values          = sv,
    base_values     = explainer.expected_value[1] if isinstance(explainer.expected_value, list)
                      else explainer.expected_value,
    data            = X_test.values,
    feature_names   = feature_names
)
shap.plots.beeswarm(shap_exp, max_display=15, show=False)
plt.tight_layout()
plt.savefig(f"{OUT}/02B_shap_beeswarm.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved SHAP beeswarm")

# ── (C) Waterfall for one rejected applicant ───────────────────────────────
# Find a high-confidence default prediction
sample_idx = np.where((y_pred == 1) & (y_test.values == 1))[0]
sample_idx = sample_idx[0] if len(sample_idx) > 0 else 2

base_val = (explainer.expected_value[1]
            if isinstance(explainer.expected_value, list)
            else explainer.expected_value)
single_exp = shap.Explanation(
    values        = sv[sample_idx],
    base_values   = base_val,
    data          = X_test.iloc[sample_idx].values,
    feature_names = feature_names
)
fig, ax = plt.subplots(figsize=(12, 7))
shap.plots.waterfall(single_exp, max_display=12, show=False)
plt.title(f"SHAP Waterfall — Rejected Applicant #{sample_idx}", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/02C_shap_waterfall_rejected.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved SHAP waterfall (applicant #{sample_idx})")


## Cell 8 · Step 7 — XAI: LIME (Local Interpretable Model-agnostic Explanations)

### Theory
LIME explains a **single prediction** by building a simple interpretable surrogate
(linear/sparse) model *around that specific data point*.

**Algorithm:**
1. Perturb the input sample (add noise / zero-out features).
2. Get predictions from the black-box model for each perturbed sample.
3. Weight perturbed samples by proximity to the original point (exponential kernel).
4. Fit a LASSO-regularised linear regression on the weighted perturbed data.
5. Coefficients of the sparse linear model = the LIME explanations.

### SHAP vs LIME comparison

| Property | SHAP | LIME |
|----------|------|------|
| Scope | Global + Local | Local only |
| Exactness | Exact (TreeExplainer) | Approximate (sampling) |
| Speed | Fast for trees | Slower (N perturbations) |
| Faithfulness | Axiomatically guaranteed | Depends on kernel/K |
| Model-agnostic | No (tree-specific) | Yes |
| Interpretable | Moderate (all features) | High (sparse K features) |

**LIME is especially useful for:**
- Explaining a specific rejection to a customer in plain language.
- Regulatory compliance (right to explanation — EU GDPR Art. 22).


In [ ]:
lime_explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data         = X_res.values,
    feature_names         = feature_names,
    class_names           = ["Good","Default"],
    mode                  = "classification",
    discretize_continuous = True,
    random_state          = 42
)

lime_exp = lime_explainer.explain_instance(
    data_row   = X_test.iloc[sample_idx].values,
    predict_fn = model.predict_proba,
    num_features = 12,
    num_samples  = 5000
)

lime_list        = lime_exp.as_list(label=1)
lime_list_sorted = sorted(lime_list, key=lambda x: abs(x[1]), reverse=True)
labels_l  = [item[0] for item in lime_list_sorted]
values_l  = [item[1] for item in lime_list_sorted]
colors_l  = ["#DC2626" if v > 0 else "#2563EB" for v in values_l]

fig, ax = plt.subplots(figsize=(12, 6))
hbars = ax.barh(range(len(labels_l)), values_l, color=colors_l,
                edgecolor="white", linewidth=0.5, height=0.65)
ax.set_yticks(range(len(labels_l)))
ax.set_yticklabels(labels_l, fontsize=9)
ax.axvline(0, color="black", lw=1)
for bar, val in zip(hbars, values_l):
    sign = "+" if val >= 0 else ""
    ax.text(val + (0.003 if val >= 0 else -0.003),
            bar.get_y() + bar.get_height()/2,
            f"{sign}{val:.4f}", va="center",
            ha="left" if val >= 0 else "right", fontsize=8)
prob_default = y_pred_proba[sample_idx]
ax.set_xlabel("LIME Weight  (positive = evidence for DEFAULT)", fontsize=11)
ax.set_title(f"LIME Local Explanation — Rejected Applicant #{sample_idx}\n"
             f"Default probability: {prob_default:.1%}", fontsize=12, fontweight="bold")
red_patch  = mpatches.Patch(color="#DC2626", label="Supports DEFAULT")
blue_patch = mpatches.Patch(color="#2563EB", label="Supports GOOD")
ax.legend(handles=[red_patch, blue_patch], loc="lower right", fontsize=9)
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(f"{OUT}/03_lime_local_explanation.png", dpi=150, bbox_inches="tight")
plt.close()
print(f"Saved LIME explanation (applicant #{sample_idx})")

print(f"\nHuman-readable LIME explanation for applicant #{sample_idx}:")
print(f"Default probability: {prob_default:.1%}\n")
for feat, weight in lime_list_sorted[:6]:
    direction = "↑ INCREASES" if weight > 0 else "↓ DECREASES"
    print(f"  {direction} default risk: {feat}")


## Cell 9 · Step 8 — Bias Audit via SHAP

### Motivation
Credit scoring models that use **protected attributes** (age, gender, nationality) as primary
decision drivers are illegal in many jurisdictions and unethical in all.

**Protected attributes checked:**
- `age` — protected under many anti-discrimination laws
- `personal_status` — encodes gender in the German Credit dataset
- `foreign_worker` — proxy for national origin

### Methodology
For each protected attribute we compute:
- Its **global SHAP rank** (where it sits among all 20 features).
- Its **mean |SHAP| value** — the average absolute impact on model output.
- A **bias verdict**: `⚠ INVESTIGATE` if its impact exceeds the 70th-percentile of all feature impacts.

### Legal context
EU GDPR Art. 22 and the Equal Credit Opportunity Act both require:
- Models must **not** use gender / nationality as primary decision drivers.
- Every rejected applicant has the **right to a human-readable explanation**.


In [ ]:
protected         = ["age", "personal_status", "foreign_worker"]
protected_present = [f for f in protected if f in feature_names]

fig, axes = plt.subplots(1, len(protected_present), figsize=(5*len(protected_present), 5))
if len(protected_present) == 1:
    axes = [axes]

for ax, feat in zip(axes, protected_present):
    fidx      = feature_names.index(feat)
    feat_shap = sv[:, fidx]
    feat_vals = X_test[feat].values

    ax.scatter(feat_vals, feat_shap, c=feat_shap, cmap="RdBu_r",
               alpha=0.6, s=20, linewidths=0)
    ax.axhline(0, color="black", lw=1)
    ax.set_xlabel(f"{feat} (encoded)", fontsize=10)
    ax.set_ylabel("SHAP Value", fontsize=10)
    rank = list(sorted_idx).index(fidx) + 1
    ax.set_title(f"'{feat}'\nGlobal rank: #{rank} / {len(feature_names)}", fontsize=10, fontweight="bold")
    ax.grid(alpha=0.3)

    mean_effect = np.abs(feat_shap).mean()
    threshold   = np.percentile(np.abs(sv).mean(axis=0), 70)
    verdict = "⚠ INVESTIGATE" if mean_effect > threshold else "✓ LOW IMPACT"
    color   = "#DC2626" if "INVESTIGATE" in verdict else "#16A34A"
    ax.text(0.05, 0.95, verdict, transform=ax.transAxes,
            color=color, fontsize=10, fontweight="bold", va="top")

fig.suptitle("Bias Audit — SHAP Dependence Plots for Protected Attributes",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(f"{OUT}/04_bias_audit_shap.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved bias audit plot\n")

for feat in protected_present:
    fidx        = feature_names.index(feat)
    mean_effect = np.abs(sv[:, fidx]).mean()
    rank        = list(sorted_idx).index(fidx) + 1
    threshold   = np.percentile(np.abs(sv).mean(axis=0), 70)
    verdict = "⚠  INVESTIGATE further" if mean_effect > threshold else "✓  LOW IMPACT"
    print(f"  {feat:<20}  rank={rank:>3}   mean|SHAP|={mean_effect:.5f}   {verdict}")


## Cell 10 · Step 9 — Project Summary Figure

A consolidated summary card is produced showing all six components of the pipeline:
Dataset → Imbalance Fix → Model → Performance → SHAP → LIME.

This figure serves as a one-page overview suitable for presentation or reporting.


In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.set_xlim(0, 12); ax.set_ylim(0, 8); ax.axis("off")

title_box = dict(boxstyle="round,pad=0.5", facecolor="#1E3A5F", edgecolor="#1E3A5F")
ax.text(6, 7.5, "Trustworthy Credit Scoring — Project Summary",
        ha="center", va="center", fontsize=14, fontweight="bold",
        color="white", bbox=title_box)

sections = [
    ("Dataset",
     f"German Credit (UCI)\n{len(df)} applicants, {len(feature_names)} features\nImbalance: ~{imbalance_ratio:.0f}:1 (Good:Default)",
     "#EFF6FF", "#2563EB"),
    ("Imbalance Fix",
     "SMOTE (k=5) + scale_pos_weight\nTraining set balanced to 1:1\nTest set untouched (no leakage)",
     "#FEF3C7", "#D97706"),
    ("Model",
     f"LightGBM (GBDT)\nn_estimators={best_iter}\nleaf-wise growth, histogram bins",
     "#F0FDF4", "#16A34A"),
    ("Performance",
     f"ROC-AUC : {roc_auc:.3f}\nAvg Prec: {ap:.3f}\nF1 Score: {f1:.3f}",
     "#FFF1F2", "#DC2626"),
    ("SHAP (Global)",
     "TreeExplainer — exact\nBeeswarm + Bar + Waterfall\nTop drivers identified",
     "#F5F3FF", "#7C3AED"),
    ("LIME (Local)",
     "LimeTabularExplainer\n5000 perturbations\nSparse linear surrogate",
     "#FFF7ED", "#EA580C"),
]

cols = 3
for i, (title, body, bg, border) in enumerate(sections):
    row = 1 - (i // cols)
    col = i % cols
    x0  = 0.2 + col * 3.9
    y0  = 1.0 + row * 3.2
    rect = mpatches.FancyBboxPatch((x0, y0), 3.5, 2.8,
                                    boxstyle="round,pad=0.15",
                                    facecolor=bg, edgecolor=border, linewidth=2)
    ax.add_patch(rect)
    ax.text(x0+1.75, y0+2.4, title, ha="center", va="top",
            fontsize=10, fontweight="bold", color=border)
    ax.text(x0+1.75, y0+1.8, body, ha="center", va="top",
            fontsize=8.5, color="#1f2937", linespacing=1.5)

plt.tight_layout()
plt.savefig(f"{OUT}/00_project_summary.png", dpi=150, bbox_inches="tight")
plt.close()

print("=" * 60)
print("  ALL DONE — output files:")
print("=" * 60)
for f in sorted(os.listdir(OUT)):
    if f.endswith(".png"):
        print(f"   📊  {f}")
